In [1]:
!pip install easyocr
!pip install streamlit 
!pip install pillow

In [2]:
import streamlit as st
import easyocr as ocr
import re
from PIL import Image

In [3]:
reader = ocr.Reader(['en'])

In [4]:
result = reader.readtext('/Users/akashravuru/Desktop/DS/Data/1.png')
print (result)

/opt/anaconda3/envs/bizcard/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


[([[np.int32(47), np.int32(113)], [np.int32(156), np.int32(113)], [np.int32(156), np.int32(157)], [np.int32(47), np.int32(157)]], 'Selva', np.float64(0.8238234915998703)), ([[np.int32(52), np.int32(158)], [np.int32(278), np.int32(158)], [np.int32(278), np.int32(190)], [np.int32(52), np.int32(190)]], 'DATA MANAGER', np.float64(0.9991661438953733)), ([[np.int32(120), np.int32(224)], [np.int32(333), np.int32(224)], [np.int32(333), np.int32(254)], [np.int32(120), np.int32(254)]], '+123-456-7890', np.float64(0.5786890937938963)), ([[np.int32(120), np.int32(262)], [np.int32(330), np.int32(262)], [np.int32(330), np.int32(294)], [np.int32(120), np.int32(294)]], '+123-456-7891', np.float64(0.601392152678678)), ([[np.int32(120), np.int32(318)], [np.int32(340), np.int32(318)], [np.int32(340), np.int32(350)], [np.int32(120), np.int32(350)]], 'WWW XYZI.com', np.float64(0.5929571230622726)), ([[np.int32(119), np.int32(355)], [np.int32(359), np.int32(355)], [np.int32(359), np.int32(391)], [np.int32(1

In [5]:
for t in result:
    text = [text[1] for text in result]

print (text)

['Selva', 'DATA MANAGER', '+123-456-7890', '+123-456-7891', 'WWW XYZI.com', 'hello@XYZ1.com', '123 ABC St , Chennai;', 'selva', 'TamilNadu 600113', 'digitals']


In [14]:
def extract_fields(text_list):
    designations = ["ceo", "manager", "director", "engineer", "developer", 
                "founder", "president", "officer", "analyst", "consultant"]
    email = ""
    phone = []
    website = ""
    name = ""
    pincode = ""
    address = []
    company_name = []
    designation = ""
    
    name = text_list[0] if text_list else ""
    for item in text_list:
        if "@" in item:
            email = item
        elif sum(c.isdigit() for c in item) > 6:
            phone.append(item)
        elif "www" in item.lower() or ".com" in item.lower():
            website = item
        elif any(title in item.lower() for title in designations):
            designation = item     
        
        else:
            match = re.search(r'\b\d{6}\b', item)
            if match:
                address.append(item)
            else:
                if item != name and any(c.isdigit() for c in item) or any(c in item for c in [',', '.', 'St', 'Rd', 'Ave', 'Nagar']):
                    address.append(item)
                elif item != name:
                    company_name.append(item)
                    
                

    
    return {
        "name": name,
        "email": email,
        "phone": ", ".join(phone),
        "website": website,
        "designation": designation,
        "address": ", ".join(address),
        "company": " ".join(company_name)
        
        
    }


{'name': 'Selva',
 'email': 'hello@XYZ1.com',
 'phone': '+123-456-7890, +123-456-7891',
 'website': 'WWW XYZI.com',
 'designation': 'DATA MANAGER',
 'address': '123 ABC St , Chennai;, TamilNadu 600113',
 'company': 'selva digitals'}

In [7]:
import sqlite3
conn = sqlite3.connect("bizcard.db")
cursor = conn.cursor()

In [29]:
cursor.execute("DROP TABLE IF EXISTS business_cards")
cursor.execute("""
    CREATE TABLE business_cards(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(50),
        email VARCHAR(50) UNIQUE,
        phone TEXT,
        company VARCHAR(50),
        designation VARCHAR(50),
        address VARCHAR(255),
        website VARCHAR(50)
    )
""")
conn.commit()

In [30]:
cursor.execute("SELECT * from business_cards")

In [31]:
def insert_card(fields):
    cursor.execute("""
                   INSERT OR IGNORE INTO business_cards(name, email, phone, company, designation, address, website) VALUES (?,?,?,?,?,?,?)
                   """,(
                       fields['name'],
                       fields['email'],
                       fields['phone'],
                       fields['company'],
                       fields['designation'],
                       fields['address'],
                       fields['website']
        ))
    conn.commit()


In [32]:
fields = extract_fields(text)
insert_card(fields)

In [34]:
cursor.execute("SELECT * FROM business_cards")
print(cursor.fetchall())

[(1, 'Selva', 'hello@XYZ1.com', '+123-456-7890, +123-456-7891', 'selva digitals', 'DATA MANAGER', '123 ABC St , Chennai;, TamilNadu 600113', 'WWW XYZI.com')]


In [38]:
cursor.execute("SELECT * FROM business_cards")
print(cursor.fetchall())

[(1, 'Selva', 'hello@XYZ1.com', '+123-456-7890, +123-456-7891', 'selva digitals', 'DATA MANAGER', '123 ABC St , Chennai;, TamilNadu 600113', 'WWW XYZI.com')]
